<a href="https://bio-pro.mintlify.app/tools/quickstart"><img align="right" src="https://img.shields.io/badge/View_in_Proto_Docs_%E2%86%92-046e7a?style=for-the-badge&logo=readthedocs&logoColor=white" alt="View in Proto Docs: Quickstart"></a>

# 01 — Getting Started with proto_tools

Welcome! This notebook shows the **universal tool pattern** used by every tool in the `proto_tools` library.

**What you'll learn**
1. The `Input` → `Config` → `run_*()` → `Output` contract every tool follows
2. How to discover tools with `ToolRegistry`
3. How to export results

This notebook uses `random_nucleotide_sample` — a pure-Python CPU tool that samples random bases at masked positions. No GPU, no downloads, no external services.

> When you're ready to run real models, jump to [02 — Tool Persistence](02_tool_persistence.ipynb).


## 1. The universal pattern

Every tool in `proto_tools` exposes exactly four things. For the `random_nucleotide_sample` tool we'll use below, they look like this:

| Component | Name | Role |
|---|---|---|
| Input class | `RandomNucleotideSampleInput` | The primary input data (sequences, structures, files) |
| Config class | `RandomNucleotideSampleConfig` | Parameters (thresholds, temperature, device). All fields have defaults. |
| Tool call | `run_random_nucleotide_sample()` | The call itself. Returns a typed output object. |
| Output class | `RandomNucleotideSampleOutput` | Results + auto-populated metadata (`execution_time`, `success`, `errors`, …) |

Every other tool follows the identical shape: swap in the tool's own class names.


In [1]:
# A tool's public surface is just these three symbols: an Input class, an optional Config class, and a run_* function.
from proto_tools.tools.mutagenesis.random_nucleotide import (
    RandomNucleotideSampleConfig,
    RandomNucleotideSampleInput,
    run_random_nucleotide_sample,
)

# 1. Build the input. Only required fields here — '_' marks positions to sample.
inputs = RandomNucleotideSampleInput(sequences=["ACGT_______ACGT", "ACGT______ACGT"])

# 2. Build a config. Every ConfigField has a default, so you override only what you care about.
#    "N" = any base (A/C/G/T); other IUPAC codes like "R" (purines) or "Y" (pyrimidines) are supported too.
config = RandomNucleotideSampleConfig(substitution_scheme="N")

# 3. Call the tool. The @tool decorator wraps the inner function with error capture + timing.
output = run_random_nucleotide_sample(inputs, config)

# 4. Output is a typed Pydantic model — tool-specific fields plus standard metadata
#    (tool_id, execution_time, timestamp, success, errors, warnings, metadata).
print("Filled sequences:", output.sequences)

Filled sequences: ['ACGTTTGCATTACGT', 'ACGTGAATAAACGT']


### Config is optional

Every field on `{Tool}Config` has a default, so you can skip it entirely:


In [2]:
# Config is optional; omit it and the @tool wrapper instantiates RandomNucleotideSampleConfig()
# with all defaults. This is the shortest possible call for any tool in the library.
output = run_random_nucleotide_sample(
    RandomNucleotideSampleInput(sequences=["ACGT________CGT"])
)
print(output.sequences)

['ACGTGTTGTGACCGT']


## 2. Discovering tools

Every tool registers itself via the `@tool` decorator. `ToolRegistry` is the index.


In [3]:
from collections import defaultdict

from proto_tools.tools import ToolRegistry

# list_all() returns ToolSpec objects — the metadata the @tool decorator captured at import time.
specs = ToolRegistry.list_all()
print(f"{len(specs)} tools registered\n")

# Bucket them by category for a readable overview.
by_category = defaultdict(list)
for spec in specs:
    by_category[spec.category].append(spec.key)

# Show up to 3 tool keys per category; collapse the rest into a "(+N more)" hint.
for category in sorted(by_category):
    keys = by_category[category]
    sample = ", ".join(keys[:3])
    more = f"  (+{len(keys) - 3} more)" if len(keys) > 3 else ""
    print(f"  {category:22s}  {sample}{more}")

122 tools registered

  binder_design           bindcraft-design, germinal-design
  causal_models           evo1-sample, evo1-score, evo2-sample  (+5 more)
  database_retrieval      alphafold-db-fetch, alphamissense-db-fetch, ccd-lookup  (+14 more)
  gene_annotation         crispr-tracr-rna, minced-crispr, miranda-scan  (+6 more)
  inverse_folding         esm-if1-sample, esm-if1-score, fampnn-sample  (+8 more)
  masked_models           ablang-embedding, ablang-gradient, ablang-sample  (+9 more)
  mutagenesis             random-nucleotide-sample, random-protein-sample
  orf_prediction          orfipy-prediction, prodigal-prediction
  rna_splicing            pangolin-predict, pangolin-score-variants, splice-transformer-prediction  (+2 more)
  sequence_alignment      blast-search, blast-create-db, mafft-align  (+4 more)
  sequence_scoring        alphagenome-predict-intervals, alphagenome-predict-sequences, alphagenome-predict-variants  (+11 more)
  structure_alignment     foldmason-msa, f

In [4]:
# Every tool advertises JSON Schemas for its Input, Config, and Output.
# Useful for auto-generating UIs, validating payloads, or exposing tools to LLMs.
schemas = ToolRegistry.get_schemas("random-nucleotide-sample")
print("Input fields: ", list(schemas["inputs"]["properties"].keys()))
print("Config fields:", list(schemas["config"]["properties"].keys()))
print("Output fields:", list(schemas["output"]["properties"].keys()))

Input fields:  ['sequences']
Config fields: ['verbose', 'device', 'timeout', 'seed', 'masking_strategy', 'substitution_scheme', 'sequence_type']
Output fields: ['tool_id', 'execution_time', 'timestamp', 'success', 'warnings', 'errors', 'metadata', 'sequences']


In [5]:
# Every tool also ships a minimal example input you can use as a template or a smoke test.
example = ToolRegistry.get_example_input("esm2-embedding")
print(type(example).__name__, "→", example.model_dump())

ESM2EmbeddingsInput → {'sequences': ['MKTL']}


## 3. Exporting results

Output objects can serialize themselves to common file formats. Check `output.output_format_options` to see what's supported for a given tool.


In [6]:
from pathlib import Path
import tempfile

# Run the tool again so we have a fresh Output to serialize.
output = run_random_nucleotide_sample(
    RandomNucleotideSampleInput(sequences=["ACGT_CGT", "G_TA_CA", "AA_CC_GG"])
)

# Each Output advertises the file formats its _export_output() implements.
# For RandomNucleotideSampleOutput that's fasta / txt / json.
print("Supported formats:", output.output_format_options)

# export() dispatches on file_format and writes to disk.
# We use a tempdir so the notebook doesn't leave files behind.
with tempfile.TemporaryDirectory() as tmp:
    output.export(name="sampled", export_path=Path(tmp), file_format="fasta")

    # Read it back to confirm what landed on disk.
    print("\n--- sampled.fasta ---")
    print((Path(tmp) / "sampled.fasta").read_text())

Supported formats: ['fasta', 'txt', 'json']

--- sampled.fasta ---
>seq_0
ACGTACGT
>seq_1
GGTATCA
>seq_2
AACCCCGG



## What's next?

- **[02 — Tool Persistence](02_tool_persistence.ipynb)** — Call GPU tools efficiently by keeping models warm between calls with `ToolInstance.persist()`.
- **[03 — Device Management](03_device_management.ipynb)** — Control which GPU your tool runs on; understand LRU eviction and CPU offload.
- **[04 — Parallel Execution](04_parallel_execution.ipynb)** — Fan out a batch of work across every GPU on the machine with `ToolPool`.

You can also browse `proto_tools/tools/{category}/{tool}/examples/example.ipynb` for a minimal example of each specific tool.
